In [1]:
import numpy as np
from scipy.special import gamma
import matplotlib.pyplot as plt

In [3]:
# Define the ψ(α, j) function
def psi(alpha, j):
    return gamma(j - alpha) / (gamma(-alpha) * gamma(j + 1))

# Define the D(α, j) diagonal matrix
def D(alpha, j):
    diag_elements = [psi(alpha_i, j) for alpha_i in alpha]
    return np.diag(diag_elements)

# Define the A_j matrix
def A_j(A, alpha, j):
    if j == 0:
        return A - np.diag(alpha)
    else:
        return -D(alpha, j + 1)

# Define G_k recursively
def G_k(A, alpha, k):
    if k == 0:
        return np.eye(A.shape[0])
    else:
        G_sum = sum(A_j(A, alpha, j) @ G_k(A, alpha, k - 1 - j) for j in range(k))
        return G_sum

# Simulate the fractional-order system
def simulate_fractional_system(A, B, alpha, x0, u, steps):
    n = A.shape[0]
    x = np.zeros((steps + 1, n))
    x[0] = x0
    
    # Calculate system response over time
    for k in range(steps):
        summation_term = sum(A_j(A, alpha, j) @ x[k - j] for j in range(k + 1))
        x[k + 1] = summation_term + B @ u[k]
    
    return x

In [4]:
# Parameters
dim_x = 5  # State dimension
dim_u = 5  # Input dimension
steps = 50  # Number of time steps
num_samples = 1000  # Number of trajectories

##### Generate multiple trajectories from one system

In [5]:
# Generate multiple trajectories 
A = np.random.rand(dim_x, dim_x) * 0.5  # Random A matrix with smaller values
B = np.random.rand(dim_x, dim_u) * 0.2  # Random B matrix with smaller values
alpha = np.random.rand(dim_x) * 0.5  # Random fractional orders in [0, 0.5]

all_trajectories = []
all_inputs = []
#all_A = []
#all_B =[]
#all_alpha = []  # Store inputs for reproducibility
for _ in range(num_samples):
    x0 = np.random.rand(dim_x)  # Random initial condition
    u = np.random.rand(steps, dim_u)  # Random input sequence
    trajectory = simulate_fractional_system(A, B, alpha, x0, u, steps)
    all_trajectories.append(trajectory)
    all_inputs.append(u)
    #all_A.append(A)
    #all_B.append(B)
    #all_alpha.append(alpha)

# Save trajectories and inputs if needed
trajectories_array = np.array(all_trajectories)
inputs_array = np.array(all_inputs)


In [9]:
# Example of saving
np.save("fractional_system_trajectories.npy", trajectories_array)
np.save("fractional_system_inputs.npy", inputs_array)
np.save("system_matrix_A.npy", A)
np.save("system_matrix_B.npy", B)
np.save("fractional_orders_alpha.npy", alpha)
# Print confirmation
print(f"Generated {num_samples} trajectories of length {steps} with dimensions:")
print(f"x: {dim_x}, u: {dim_u}")
print(f"Saved A, B, and alpha for reproducibility.")

Generated 1000 trajectories of length 50 with dimensions:
x: 5, u: 5
Saved A, B, and alpha for reproducibility.


##### Generate multiple trajectories from different systems

In [4]:
# Generate multiple trajectories 
all_trajectories = []
all_inputs = []
all_A = []
all_B =[]
all_alpha = []  # Store inputs for reproducibility
for _ in range(num_samples):
    A = np.random.rand(dim_x, dim_x) * 0.5  # Random A matrix with smaller values
    B = np.random.rand(dim_x, dim_u) * 0.2  # Random B matrix with smaller values
    alpha = np.random.rand(dim_x) * 0.5  # Random fractional orders in [0, 0.5]
    x0 = np.random.rand(dim_x)  # Random initial condition
    u = np.random.rand(steps, dim_u)  # Random input sequence
    trajectory = simulate_fractional_system(A, B, alpha, x0, u, steps)
    all_trajectories.append(trajectory)
    all_inputs.append(u)
    all_A.append(A)
    all_B.append(B)
    all_alpha.append(alpha)

In [5]:
# Save trajectories and inputs if needed
trajectories_array = np.array(all_trajectories)
inputs_array = np.array(all_inputs)
A_array = np.array(all_A)
B_array = np.array(all_B)
alpha_array = np.array(all_alpha)

In [6]:
# Example of saving
np.save("fractional_system_trajectories.npy", trajectories_array)
np.save("fractional_system_inputs.npy", inputs_array)
np.save("system_matrix_A.npy", A_array)
np.save("system_matrix_B.npy", B_array)
np.save("fractional_orders_alpha.npy", alpha_array)
# Print confirmation
print(f"Generated {num_samples} trajectories of length {steps} with dimensions:")
print(f"x: {dim_x}, u: {dim_u}")
print(f"Saved A, B, and alpha for reproducibility.")

Generated 1000 trajectories of length 50 with dimensions:
x: 5, u: 5
Saved A, B, and alpha for reproducibility.


##### generate optimal control 

In [20]:
# Define the ψ(α_i, j) function
def psi(alpha_i, j):
    """
    Compute ψ(α_i, j) = Γ(j - α_i) / (Γ(-α_i) Γ(j + 1)).

    Parameters:
    - alpha_i: Scalar value of α_i (float).
    - j: Scalar index j (integer).

    Returns:
    - ψ(α_i, j): Computed value (float).
    """
    return gamma(j - alpha_i) / (gamma(-alpha_i) * gamma(j + 1))

# Define the diagonal matrix D(α, j)
def D(alpha, j):
    """
    Compute the diagonal matrix D(α, j).

    Parameters:
    - alpha: List or numpy array of α values.
    - j: Index j (integer).

    Returns:
    - D: Diagonal matrix (numpy array).
    """
    n = len(alpha)
    diag_elements = [psi(ai, j) for ai in alpha]
    return np.diag(diag_elements)

# Define A_j based on the given formula
def A_j(alpha, j, A):
    """
    Compute A_j based on the given formula:
    A_j = A - diag(α_1, ..., α_n) for j = 0
    A_j = -D(α, j + 1) for j >= 1

    Parameters:
    - alpha: List or numpy array of α values.
    - j: Index j (integer).
    - A: Matrix A (numpy array).

    Returns:
    - A_j: Computed matrix A_j (numpy array).
    """
    if j == 0:
        diag_matrix = np.diag(alpha)  # Create diag(α_1, ..., α_n)
        return A - diag_matrix
    else:
        return -D(alpha, j + 1)

# (A_0^T)
def A_j_T(alpha, j, A):
    """
    Compute A_j based on the given formula:
    A_j = A - diag(α_1, ..., α_n) for j = 0
    A_j = -D(α, j + 1) for j >= 1

    Parameters:
    - alpha: List or numpy array of α values.
    - j: Index j (integer).
    - A: Matrix A (numpy array).

    Returns:
    - A_j: Computed matrix A_j (numpy array).
    """
    if j == 1:
        diag_matrix = np.diag(alpha)  # Create diag(α_1, ..., α_n)
        return (A - diag_matrix).T
    else:
        return -D(alpha, j)
    
def A_j_list(alpha, A, T):
    """
    Generate a list of A_j^T matrices for j = 0 to T-1.

    Parameters:
    - alpha: List or numpy array of α values.
    - T: Total number of A_j matrices to compute.
    - A: Matrix A (numpy array).

    Returns:
    - A_j_list: List of A_j^T matrices.
    """
    return [A_j_T(alpha, j+1, A) for j in range(T)]
'''''''''
def A_mlist(alpha, A, T):
    """
    Generate a list of A_j^T matrices for j = 0 to T-1.

    Parameters:
    - alpha: List or numpy array of α values.
    - T: Total number of A_j matrices to compute.
    - A: Matrix A (numpy array).

    Returns:
    - A_j_list: List of A_j^T matrices.
    """
    return [A_j(alpha, j, A) for j in range(T)]
'''''''''''
# Define G_k using recursion
def G_k(alpha, A, max_k):
    """
    Compute G_k recursively based on the given formula:
    G_k = I for k = 0
    G_k = Σ_{j=0}^{k-1} A_j G_{k-1-j} for k >= 1

    Parameters:
    - alpha: List or numpy array of α values.
    - k: Current recursion depth (integer).
    - A: Matrix A (numpy array).
    - max_k: Maximum k for recursion.

    Returns:
    - G_list: List of G_k matrices up to G_max_k (list of numpy arrays).
    """
    n = len(alpha)
    I = np.eye(n)  # Identity matrix
    G_list = [I]  # Initialize G_0 as I

    for current_k in range(1, max_k + 1):
        G_k = np.zeros_like(A)  # Initialize G_k to zero
        for j in range(current_k):
            A_j_matrix = A_j(alpha, j, A)
            G_k += A_j_matrix @ G_list[current_k - 1 - j]  # Recursive formula
        G_list.append(G_k)

    return G_list

In [21]:
def create_G_lambda(T, Q, B, R, A_j_list, G_list):
    """
    Generate the G_lambda matrix as shown in the provided image.

    Parameters:
    - T: int, time horizon
    - Q: numpy array, a square matrix
    - B: numpy array, input matrix
    - R: numpy array, a square matrix
    - A_j_list
    - G_list

    Returns:
    - G_lambda: numpy array, the generated G_lambda matrix
    """
    # Get dimensions
    n, m = B.shape
    BR_inv_B_T = B @ np.linalg.inv(R) @ B.T 

    # Initialize the matrix
    G_lambda = np.zeros((T * n, T * n))

    # Fill in the blocks
    for row in range(T):
        for col in range(T):
            if row > col :
                G_lambda[row * n:(row + 1) * n, col * n:(col + 1) * n] = -Q @ G_list[row - col] @ BR_inv_B_T
            elif row == col:
                G_lambda[row * n:(row + 1) * n, col * n:(col + 1) * n] = -Q @ BR_inv_B_T               #G_0 = I
            elif row < col:
                G_lambda[row * n:(row + 1) * n, col * n:(col + 1) * n] = A_j_list[col - row - 1]
    
    return G_lambda

In [22]:
def generate_symmetric_matrix(n):
    M = np.random.rand(n, n)
    return M.T @ M  

In [23]:
def create_H_lambda(T, Q, G_list):
    """
    Generate the H_lambda matrix as shown in the provided structure.

    Parameters:
    - T: int, time horizon (number of blocks in the matrix).
    - Q: numpy array, a square matrix.
    - G_list

    Returns:
    - H_lambda: numpy array, the generated H_lambda matrix.
    """
    # Get the dimensions of Q
    n = Q.shape[0]

    # Initialize the H_lambda matrix with zeros
    H_lambda = np.zeros((T * n, n))

    # Fill in the rows of H_lambda
    for i in range(T):
        G_i = G_list[i + 1]  # Compute G_i (1-based index)
        H_lambda[i * n:(i + 1) * n, :] = Q @ G_i

    return H_lambda

def compute_lambda(G_lambda, H_lambda, x_0):
    # Identity matrix of the same size as G_lambda
    I = np.eye(G_lambda.shape[0])

    # Compute (I - G_lambda)^(-1)
    inv_I_minus_G_lambda = np.linalg.inv(I - G_lambda)

    # Compute λ
    lambda_vector = 2 * inv_I_minus_G_lambda @ H_lambda @ x_0

    return lambda_vector

In [24]:
def compute_u_with_block_matrix(R, B, G_lambda, H_lambda, x0, T):
    """
    Compute u using the formula:
    u = -[R^(-1) B^T, R^(-1) B^T, ..., R^(-1) B^T] @ ((I - G_lambda)^(-1) H_lambda x0)

    Parameters:
    - R: numpy array, square matrix R (assumed invertible).
    - B: numpy array, matrix B.
    - G_lambda: numpy array, square matrix G_lambda.
    - H_lambda: numpy array, matrix H_lambda.
    - x0: numpy array, vector x0.
    - T: int, number of repetitions of the block matrix.

    Returns:
    - u: numpy array, the computed vector u.
    """
    # Compute R^(-1)
    R_inv = np.linalg.inv(R)

    # Compute R^(-1) B^T
    R_inv_B_T = R_inv @ B.T

    # Construct the block matrix [R^(-1) B^T, R^(-1) B^T, ..., R^(-1) B^T] (T times)
    I_T = np.eye(T)
    block_diag_matrix = np.kron(I_T, R_inv_B_T)

    # Compute (I - G_lambda)^(-1)
    I = np.eye(G_lambda.shape[0])
    inv_I_minus_G = np.linalg.inv(I - G_lambda)

    # Compute (I - G_lambda)^(-1) * H_lambda * x0
    term = inv_I_minus_G @ H_lambda @ x0

    # Compute u
    u_vector = -block_diag_matrix @ term
    return u_vector

In [25]:
all_Q = []
all_R = []
all_U = []
max_k = 50
T = max_k
for i in range(num_samples):
    x_0 = trajectories_array[i, 0]  # Initial state from trajectory
    Q = generate_symmetric_matrix(A.shape[0])  # Random symmetric matrix Q
    R = generate_symmetric_matrix(B.shape[1])  # Random symmetric matrix R
    
    G_list = G_k(alpha, A, max_k)
    A_jlist = A_j_list(alpha, A, T)
    
    G_lambda_matrix = create_G_lambda(T, Q, B, R, A_jlist, G_list)
    H_lambda_matrix = create_H_lambda(T, Q, G_list)
    
    U = compute_u_with_block_matrix(R, B, G_lambda_matrix, H_lambda_matrix, x_0, T)
    
    all_U.append(U)
    all_Q.append(Q)
    all_R.append(R)


In [39]:
# Save trajectories and inputs if needed
U_array = np.array(all_U).reshape(num_samples, T, dim_u)
Q_array = np.array(all_Q)
R_array = np.array(all_R)

In [44]:
# Example of saving
np.save("optimal_control_U.npy", U_array)
np.save("LQR_Q.npy", Q_array)
np.save("LQR_R.npy", R_array)